In [2]:
from camel_tools.disambig_mle import MLEDisambiguator
from camel_tools.morphology.database import MorphologyDB
from camel_tools.morphology.analyzer import Analyzer
import pandas as pd

class OptimizedCamelStemmer:
    def __init__(self):
        # تحميل قاعدة البيانات والمحلل
        self.db = MorphologyDB.builtin_db()
        self.analyzer = Analyzer(self.db)
        # إضافة ميزة "إزالة اللبس" لتحسين الاختيار
        self.disambiguator = MLEDisambiguator.builtin_disambiguator()

    def get_best_root(self, sentence):
        # تقسيم الجملة إلى كلمات
        words = sentence.split()
        
        # استخدام الـ Disambiguator لفهم السياق
        disambig_results = self.disambiguator.disambiguate(words)
        
        final_results = []
        for disambig in disambig_results:
            # الحصول على أفضل تحليل صرفي للكلمة
            best_analysis = disambig.scored_analyses[0].analysis
            
            final_results.append({
                'الكلمة': disambig.word,
                'الجذر المحسن': best_analysis.get('root', 'N/A'),
                'التشكيل الصحيح': best_analysis.get('diac', 'N/A'),
                'الثقة (Score)': disambig.scored_analyses[0].score
            })
        
        return pd.DataFrame(final_results)

# --- تجربة التحسين ---
try:
    optimizer = OptimizedCamelStemmer()
    sentence = "المسافرون يتحدثون عن استخراج البيانات"
    df = optimizer.get_best_root(sentence)
    print(df.to_markdown(index=False))
except Exception as e:
    print(f"حدث خطأ (ربما بسبب نقص البيانات): {e}")

In [ ]:
# ما الذي تم تحسينه هنا؟
# إزالة اللبس (Disambiguation): بدلاً من أن تخمن الخوارزمية الجذر، تقوم أداة MLEDisambiguator بمقارنة الكلمة بما قبلها وما بعدها.

# مثال: كلمة "مدرس" قد تكون "مُدَرِّس" (جذر: درس) أو "مَدْرَسَة" (جذر: درس). المحسن يختار الأنسب للجملة.

# نظام الثقة (Scoring): أضفنا عمود score لمعرفة مدى تأكد الخوارزمية من الناتج. إذا كان الرقم منخفضاً، يمكنك تحويل الكلمة لـ ISRI كخيار احتياطي.

# معالجة الجمل وليس الكلمات: التحسين الحقيقي في CAMel Tools يظهر عند إدخال جملة كاملة وليس كلمات منفصلة.